In [6]:
# !pip install transformers torch accelerate

In [5]:
import warnings
import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

#  CPU OPTIMIZATION

torch.set_num_threads(4)  # Adjust based on your CPU (try 2–8)

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)


#  BEST MODEL FOR CPU

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = torch.device("cpu")

print("=" * 60)
print(f"Model  : {MODEL_NAME}")
print(f"Device : {device}")
print("=" * 60)
print("Loading model...\n")

# LOAD TOKENIZER & MODEL

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.to(device)
model.eval()

print("✅ Model loaded successfully!\n")


# RESPONSE GENERATION (FAST)

def generate_response(user_input, history):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Answer in short and clear sentences."
        }
    ] + history + [{"role": "user", "content": user_input}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,        
            do_sample=True,
            temperature=0.6,          
            top_p=0.85,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_tokens = outputs[0][input_len:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=True)

    return response.strip() if response else "Try asking differently."


# CHAT LOOP

def run_chatbot():
    print(" Chatbot: Hello! I am your AI assistant.")
    print("-" * 50)
    print("Type 'exit' to quit\n")

    history = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nChatbot: Session ended. Goodbye!")
            break

        if not user_input:
            continue

        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: Goodbye! 👋")
            break

        response = generate_response(user_input, history)

        # Save conversation
        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": response})

        #  LIMIT HISTORY (IMPORTANT FOR SPEED)
        history = history[-10:]

        print(f"Chatbot: {response}\n")


# RUN

if __name__ == "__main__":
    run_chatbot()

Model  : Qwen/Qwen2.5-0.5B-Instruct
Device : cpu
Loading model...



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Model loaded successfully!

 Chatbot: Hello! I am your AI assistant.
--------------------------------------------------
Type 'exit' to quit



You:  HELLO


Chatbot: Hello! How can I assist you today?



...

Chatbot: Python was created by Guido van Rossum, an English programmer who started working on it in 1989 at the University of Cambridge. He envisioned it as a general-purpose programming language with a simple syntax and a focus on efficiency. The first version of Python was released in 19



You:  Thank you


Chatbot: You're welcome! If you have any more questions, feel free to ask.



You:  exit


Chatbot: Goodbye! 👋
